# BIL458 Computer Vision — Term Project

**Title:** CancerAI: A Comparative Study of Deep Learning Strategies for Multi-Modal Medical Image Classification

**Author:** Farah Alhasan (21120205709)

**Instructor:** Assoc. Prof. Murat Gezer

This notebook records the experiments for four separate classification tasks: brain MRI, breast histopathology, skin lesions, and lung CT. The labels are tumour and lesion categories taken from the source datasets. The study compares training strategies. It is not a universal cancer detector, and it is not a clinical device.

The 48 runs are already saved under `results/`. `RUN_FULL_STUDY` remains `False`, so Restart & Run All loads those scores, the figures, and the unit tests. It does not train again. The written report is `deliverables/BIL458_CancerAI_Report.pdf`. The presentation is `deliverables/BIL458_CancerAI_Slides.pptx`.

The demonstration page is https://43a3d99492816eb041.gradio.live. The same application can be started locally with `python app.py`, at http://127.0.0.1:7860.

## 0. Environment and reproducibility

The question is whether the choice of CNN training strategy, and the use of training augmentation, changes test macro-F1, and whether that change is the same on all four modalities.

- H1. Fine-tuned ResNet18 outperforms both a custom CNN trained from scratch and a frozen ResNet18.
- H2. Training-time augmentation improves generalisation, particularly when the classes are imbalanced.
- H3. The size of the gain depends on the modality.
- H4. Accuracy conceals errors on minority classes. Macro-F1 and per-class recall make those errors visible.

Training seeds are 42, 123, and 2026. The split is fixed with seed 42 and is not redrawn when the training seed changes. Each reported score is the mean and the sample standard deviation across the three seeds.

On Google Colab the notebook is opened without the local project tree. The first code cell clones [CancerAI](https://github.com/FarahBebs/CancerAI) and installs the required libraries. The image collections are not in the repository. Cells that need those files report the absence and continue. Saved metrics and figures still load.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def find_project(start):
    for candidate in (start, start.parent, Path("/content/CancerAI")):
        if (candidate / "cancerai" / "__init__.py").exists():
            return candidate.resolve()
    return None

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

ROOT = find_project(Path.cwd().resolve())
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/CancerAI")
    if not (ROOT / "cancerai" / "__init__.py").exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/FarahBebs/CancerAI.git",
            str(ROOT),
        ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "numpy", "pillow", "scikit-learn", "medmnist", "tqdm",
    ])
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "pillow-heif"])

if ROOT is None or not (ROOT / "cancerai" / "__init__.py").exists():
    raise FileNotFoundError(
        "The cancerai package is not here. On Colab, run this cell with internet access. "
        "On your computer, open the notebook from the CancerAI project folder."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project", ROOT)
print("colab", IN_COLAB)


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import platform
import torch
from cancerai.training import get_device, seed_everything

SEED = 42
seed_everything(SEED)
print("python", platform.python_version())
print("torch", torch.__version__)
print("device", get_device())
print("project", ROOT)


## 1. Datasets

Split membership is written once to `results/splits/` and reloaded by every later run. That is the control this study can enforce.

- **Breast histopathology (BreakHis).** The patient identifier is taken from the folder name. A patient who appears in more than one split is rejected. This is the only patient-level split in the study.
- **Skin lesions (DermaMNIST / HAM10000).** The official MedMNIST split is used. Lesion identifiers are not present in the released file, so independence at the lesion level is not claimed.
- **Brain MRI and lung CT.** The public filenames were inspected. They do not provide a patient or scan identifier. Those results are therefore not described as patient-level.

In [ ]:
from cancerai.datasets import audit_datasets, describe, prepare

try:
    report = audit_datasets()
except (FileNotFoundError, RuntimeError) as exc:
    report = None
    print(exc)
    print("The image collections are not in this runtime. The saved scores below do not need them.")

if report is not None:
    print("data root:", report["data_root"])
    for name, entry in report["datasets"].items():
        print("\n==", name, "==")
        print("patient_independent:", entry["patient_independent"])
        print(entry["leakage_note"])
        for split, counts in entry["counts"].items():
            print(f"  {split}: {sum(counts.values())}")


In [ ]:
# One unaugmented bundle per modality. The same split file feeds every model.
DATA_READY = True
for name in ("brain", "breast", "skin", "lung"):
    try:
        bundle = prepare(name, augment=False)
    except (FileNotFoundError, RuntimeError, OSError) as exc:
        DATA_READY = False
        print(name, "skipped:", exc)
        continue
    print(describe(bundle))
    print()


## 2. Preprocessing and augmentation

Every model receives a 224×224 RGB image, normalised with the ImageNet mean and standard deviation. The input pipeline is the same for every configuration, so differences in score are not an artefact of preprocessing.

Training augmentation is modality-specific:

- **Breast histopathology:** horizontal and vertical flips, rotation of ±20°, and mild brightness and contrast changes. Hue is not altered, because stain colour is informative.
- **Skin lesions:** horizontal flip, rotation of ±15°, and mild changes in brightness, contrast, and saturation. Hue is not altered.
- **Brain MRI:** rotation of ±10°. Left–right flips are not used.
- **Lung CT:** rotation of ±8°. Flips and colour changes are not used.

Validation and test images are resized and normalised only. They are never augmented.

In [ ]:
import matplotlib.pyplot as plt
from cancerai.datasets import build_transform, prepare

def names(dataset, train, augment):
    transform = build_transform(dataset, train=train, augment=augment)
    return [type(step).__name__ for step in transform.transforms]

for dataset in ("brain", "breast", "skin", "lung"):
    print(dataset, "train", names(dataset, True, True))
print("eval", names("brain", False, False))

try:
    bundle = prepare("brain", augment=False)
    fig, axes = plt.subplots(1, 4, figsize=(10, 2.6))
    for axis, index in zip(axes, range(4)):
        axis.imshow(bundle.train_ds.raw(index))
        axis.set_title(bundle.classes[int(bundle.train_ds.rows[index]["label"])], fontsize=8)
        axis.axis("off")
    fig.suptitle("Brain MRI training examples (no augmentation)")
    plt.show()
except (FileNotFoundError, RuntimeError, OSError) as exc:
    print("Example images skipped:", exc)


## 3. Baseline

The baseline is a four-block convolutional network trained from scratch. Each block is a convolution, a ReLU, and max pooling, followed by dropout and a linear classifier. The network is not initialised from ImageNet.

A convolution reads a local patch,

$$Y(i,j)=\sum_{u=0}^{k-1}\sum_{v=0}^{k-1} X(i+u,j+v)\,K(u,v)$$

and the ReLU is $f(x)=\max(0,x)$. One convolutional layer costs about $O(HW C_{in} C_{out} K^2)$ operations. The custom network uses four convolutions. ResNet18 repeats a residual form of the same operation through a much deeper backbone, which accounts for its larger parameter count and higher cost. Freezing the backbone trains only the new classification head.

## 4. Proposed method

Training uses weighted cross-entropy. For a one-hot label,

$$L=-\sum_{c=1}^{C} w_c y_c \log \hat y_c$$

where $w_c = N / (C n_c)$, rescaled so that the weights average to 1. The network returns logits, and the softmax is applied inside the loss. The implementation is the mean of $-w_y \log \mathrm{softmax}(z)_y$. It is not `CrossEntropyLoss` divided by the sum of the class weights.

The primary metric, macro-F1, gives every class equal weight:

$$F1_{macro}=\frac{1}{C}\sum_{c=1}^{C}\frac{2 P_c R_c}{P_c+R_c}$$

The ResNet18 comparison has three settings: a frozen backbone, fine-tuning with augmentation, and fine-tuning without augmentation. Where ROC-AUC is defined, it is one-versus-rest with macro averaging.

In [ ]:
from cancerai.models import build_model, count_params

for name, freeze, pretrained in (
    ("custom_cnn", False, False),
    ("resnet18", True, False),
    ("resnet18", False, False),
):
    model = build_model(name, n_classes=4, freeze_backbone=freeze, pretrained=pretrained)
    total, trainable = count_params(model)
    label = "frozen head only" if freeze else name
    print(f"{label:22}  total {total:10,d}  trainable {trainable:10,d}")


## 5. Training

All configurations share the loop in `cancerai.training.train_model`: AdamW with weight decay $10^{-4}$, cosine decay of the learning rate to $10^{-6}$, and a batch size of 32. The checkpoint retained for testing is the epoch with the highest validation macro-F1. Early stopping uses a patience of 3 and a minimum improvement of 0.001. The test set is evaluated only after that checkpoint has been restored.

| Configuration | Epoch cap | Learning rate |
|---|---:|---:|
| Custom CNN | 10 | $10^{-3}$ |
| Frozen ResNet18 | 8 | $10^{-3}$ |
| Fine-tuned ResNet18, with augmentation | 8 | $10^{-4}$ |
| Fine-tuned ResNet18, without augmentation | 8 | $10^{-4}$ |

In [ ]:
RUN_FULL_STUDY = False  # The 48 runs are already saved. True trains any run that has no metrics file.

if RUN_FULL_STUDY:
    from experiments.run_experiments import main
    main([])
else:
    print("Using the saved 48 runs in results/metrics.")
    print("Set RUN_FULL_STUDY = True only to retrain. A finished metrics file is skipped unless --force is passed to the script.")


## 6. Evaluation

Each configuration is trained with three seeds. The reported value is the mean ± the sample standard deviation.

The primary metric is test macro-F1. The secondary record includes accuracy, macro precision, macro recall, the confusion matrix, one-versus-rest macro ROC-AUC when every required class is present in the split, training time, and the time for a single forward pass.

In [ ]:
from experiments.summarize import comparison_tables, load_runs

runs = load_runs()
print(comparison_tables(runs))
print("metric files found:", len(runs))


In [ ]:
from IPython.display import Image, Markdown, display

comparison = ROOT / "results" / "plots" / "macro_f1_comparison.png"
display(Markdown("**Figure.** Test macro-F1 by modality and training strategy. The printed table reports the mean and the sample standard deviation."))
display(Image(filename=str(comparison)))

## 7. Ablation

The ablation compares fine-tuning with augmentation against fine-tuning without it. Both settings use ResNet18 at the same learning rate, on the same split, and with the same seeds. Only the training transform changes. Validation and test images remain unaugmented, so a higher training accuracy is not treated as an improvement unless test macro-F1 also changes.

## 8. Failure analysis

Each run writes its highest-confidence test errors to `results/plots/` and a Grad-CAM figure to `results/gradcam/`. Three results from the fine-tuned, augmented ResNet18 are not visible in the mean score.

- **BreakHis.** Phyllodes tumour has an F1 of 0.007 ± 0.012. About 143 of 235 test slides are assigned to fibroadenoma. The two benign fibroepithelial patterns are visually close. Phyllodes receives the highest class weight, about 4.1, because the training split contains only 60 slides, and the class is still missed. Grad-CAM on these errors lies on the tissue rather than on the image border, which is consistent with confusion between similar categories rather than a border artefact. Ductal carcinoma is also confused with lobular carcinoma.
- **DermaMNIST.** Melanoma has an F1 of 0.568, with precision 0.484 and recall 0.691. On average, about 140 nevi are labelled melanoma. Nevi account for 1,341 of 2,005 test images, so accuracy (0.787 ± 0.010) is higher than macro-F1 (0.719 ± 0.014).
- **Brain MRI and lung CT.** The scores are high, but the filenames do not identify the patient. A high brain score is not evidence that slices from the same examination were kept apart. The benign lung class rests on 18 test slices.

The demonstration page is a teaching interface. A score shown there is not evidence that a prediction is clinically correct. Grad-CAM indicates which spatial features moved the winning logit. It is not a tumour outline.

In [ ]:
import numpy as np
from cancerai.metrics import worst_mistakes

# Shape of the failure picker, on a toy prediction vector.
y_true = np.array([0, 1, 1, 0])
y_pred = np.array([0, 0, 1, 1])
probs = np.array([[0.9, 0.1], [0.8, 0.2], [0.4, 0.6], [0.3, 0.7]])
print("highest-confidence mistakes:", worst_mistakes(y_true, y_pred, probs, k=5))


In [ ]:
display(Markdown("**Figure.** Confusion matrix for BreakHis, fine-tuned ResNet18 with augmentation, seed 42."))
display(Image(filename=str(ROOT / "results" / "plots" / "breast__finetune_aug__seed42_confusion.png")))
display(Markdown("**Figure.** Grad-CAM for high-confidence BreakHis errors, seed 42. The map is not a tumour outline."))
display(Image(filename=str(ROOT / "results" / "gradcam" / "breast__finetune_aug__seed42.png")))
display(Markdown("**Figure.** Confusion matrix for DermaMNIST, fine-tuned ResNet18 with augmentation, seed 42."))
display(Image(filename=str(ROOT / "results" / "plots" / "skin__finetune_aug__seed42_confusion.png")))

## 9. Unit tests

The tests check the weighted-loss formula, the patient-level split, the augmentation policy, the definition of macro-F1, and the parameter counts. They do not require the image collections or a trained checkpoint.

In [ ]:
import importlib.util
import sys
import unittest
from pathlib import Path

def project_root():
    here = Path.cwd().resolve()
    for candidate in (here, here.parent, Path("/content/CancerAI")):
        if (candidate / "tests" / "test_core.py").exists():
            return candidate.resolve()
    return None

root = project_root()
if root is None:
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise FileNotFoundError(
            "tests/test_core.py is not in this folder. Open the notebook from the CancerAI project."
        ) from exc
    import subprocess
    root = Path("/content/CancerAI")
    if not (root / "tests" / "test_core.py").exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/FarahBebs/CancerAI.git",
            str(root),
        ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pillow", "scikit-learn",
    ])

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

spec = importlib.util.spec_from_file_location("cancerai_test_core", root / "tests" / "test_core.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
suite = unittest.defaultTestLoader.loadTestsFromModule(module)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Unit tests failed"
print("Tests passed")


## 10. Demonstration

The public page and the local page are the same application. A new visit opens on a breast histopathology example, with the predicted category, the coarser group, the category scores, and the Grad-CAM map already displayed.

https://43a3d99492816eb041.gradio.live

The same page is served locally at http://127.0.0.1:7860 after `python app.py`. The public address is a temporary tunnel to that local process. It remains available for up to about one week, and only while the process is running.

![Demonstration page opened on the breast histopathology example](https://raw.githubusercontent.com/FarahBebs/CancerAI/main/notebooks/figures/site-full.png)

The coarser group is obtained by adding the category probabilities according to a fixed map. It is not a second trained model. For brain MRI the groups are tumour and no tumour; a tumour is not reported as cancer, because the source labels do not distinguish benign from malignant. For breast histopathology the groups are benign and malignant. For skin the groups are cancer-related and not cancer-related; actinic keratoses are precancerous and are placed in the cancer-related group only because that is the HAM10000 convention. For lung CT the groups are malignant and not malignant.

Each demonstration checkpoint is the fine-tuned ResNet18 with augmentation that obtained the best validation macro-F1 among the three seeds. The weights are `models/brain.pt`, `models/breast.pt`, `models/skin.pt`, and `models/lung.pt`.

## Conclusion

Fine-tuning ResNet18 is the strongest of the three training strategies on every modality. With augmentation, test macro-F1 is 0.989 ± 0.004 on brain MRI, 0.506 ± 0.031 on breast histopathology, 0.719 ± 0.014 on skin, and 0.946 ± 0.021 on lung CT. A frozen backbone is a useful intermediate result on brain MRI and lung CT. On skin it falls below the custom CNN. Augmentation improves the patient-level breast result and lowers the skin and lung results.

H1 and H3 are supported. H2 is supported for breast histopathology only. H4 is supported for breast and skin: accuracy conceals the missed phyllodes class and the over-called melanoma class.

Only BreakHis supports a patient-level claim, and on that collection the headline macro-F1 is 0.506 ± 0.031. Brain MRI and lung CT remain in the comparison with that limitation stated. A follow-up experiment should replace those two collections with a release that provides a scan identifier, or should recover HAM10000 lesion identifiers for DermaMNIST before treating a test lesion as unseen. Until then, the demonstration page is a teaching tool rather than a clinical result.